# Experiment 1 — Domain 1 (Geometry): Phase 1 Dataset Generation

Generates the **300-polygon, 9-property** geometry dataset for Experiment 1
(`serialization_experiment_1.pdf`, Section 3).

This extends the serialization **pilot** (90 polygons, 7 properties). The
generator logic, validity constraints, and serialization are **unchanged** —
only the scale (90 → 300) and the property set (7 → 9) differ.

**Per the PDF (Section 3):**

| Tier | Count | Vertices | Coordinates |
|------|-------|----------|-------------|
| simple | 100 | 3–8  | integers in [0, 100] |
| medium | 100 | 10–20 | integers in [0, 100] |
| hard   | 100 | 20–40 | floats (2dp) in [0, 1000] |

Within each tier: 3 shape categories (convex / concave / irregular), ~33 each
(approx balanced, exact balance not required).

**9 properties** (Table 4): vertex_count, bbox, centroid, area, perimeter,
convex, orientation *(the 7 pilot properties)* **+ aspect_ratio,
edge_length_variance** *(new in Experiment 1, both global, rel-error eval)*.

Output:
- `geometry_exp1_dataset.json` — 300 polygons with WKT + ground truth
- `geometry_exp1_summary.json` — summary statistics (Section 7)

Random seed fixed at **42** and recorded in every record.


In [ ]:
# Phase 0: environment
!pip install shapely matplotlib --quiet

import json, math, random
import shapely
from shapely.geometry import Polygon, MultiPoint
from shapely import affinity
import matplotlib.pyplot as plt

print("shapely", shapely.__version__)

## 1. Core helpers

Coordinate rounding, the convexity test, and the Section 3.2 / 4.4 validity
checker. Identical to the pilot.


In [ ]:
# Round coordinates: integers for simple/medium, 2dp for hard.
def round_coords(coords, coord_type):
    if coord_type == "integer":
        return [(round(x), round(y)) for x, y in coords]
    else:  # "float_2dp"
        return [(round(x, 2), round(y, 2)) for x, y in coords]


# Convexity test: a polygon equals its own convex hull only if it has no dents.
# normalize() makes the comparison ignore vertex order / starting point.
def is_convex(poly):
    hull = poly.convex_hull
    return poly.normalize().equals(hull.normalize())


# Validity checker: every rule from the validity-constraints section.
# Returns (True, None) if acceptable, or (False, reason) if rejected.
def check_validity(poly, vmin, vmax, bound_lo, bound_hi):
    if poly is None or poly.is_empty:
        return False, "empty"
    if not poly.is_valid:                       # rule 1
        return False, "not_valid"
    if not poly.is_simple:                      # rule 2
        return False, "not_simple"
    if poly.area <= 10:                         # rule 3
        return False, "area_too_small"

    coords = list(poly.exterior.coords)[:-1]    # drop repeated closing coord
    n = len(coords)

    if not (vmin <= n <= vmax):                 # rule 6
        return False, "vertex_count_out_of_range"

    for x, y in coords:                         # rule 7
        if not (bound_lo <= x <= bound_hi and bound_lo <= y <= bound_hi):
            return False, "out_of_bounds"

    for i in range(n):                          # rule 4
        if coords[i] == coords[(i + 1) % n]:
            return False, "duplicate_adjacent"

    for i in range(n):                          # rule 5
        ax, ay = coords[i]
        bx, by = coords[(i + 1) % n]
        cx, cy = coords[(i + 2) % n]
        cross = (bx - ax) * (cy - ay) - (by - ay) * (cx - ax)
        if abs(cross) < 1e-9:
            return False, "collinear"

    # rule 5b: reject sliver shapes (degenerate near-line polygons).
    minx = min(x for x, y in coords); maxx = max(x for x, y in coords)
    miny = min(y for x, y in coords); maxy = max(y for x, y in coords)
    bbox_area = (maxx - minx) * (maxy - miny)
    if bbox_area <= 0 or (poly.area / bbox_area) < 0.05:
        return False, "sliver_low_fill"

    return True, None


print("Core helpers defined.")

## 2. Tiers and shape generators

Generation methods (Section 3.2, same as the pilot):
- **convex** — convex hull of random points (Valtr for the hard tier, which
  needs 20–40 vertices a random hull can't reliably reach).
- **concave** — radial / star method, `r_min/r_max ∈ [0.4, 0.8]` for mild
  concavity.
- **irregular** — radial base + affine stretch (`s ∈ [2, 5]` on one axis) +
  random rotation + translate into bounds.


In [ ]:
# Each tier: vertex range, coordinate bounds, coordinate type, and the
# point-count range used by the hull-based convex generator.
TIERS = {
    #         vmin vmax  lo   hi    coord_type     m_range
    "simple": (3,   8,   0,   100,  "integer",     (10, 30)),
    "medium": (10,  20,  0,   100,  "integer",     (30, 80)),
    "hard":   (20,  40,  0,   1000, "float_2dp",   (60, 150)),
}


# Convex generator: convex hull of random points (simple/medium tiers).
def gen_convex(rng, vmin, vmax, lo, hi, coord_type, m_range, max_tries=3000):
    for _ in range(max_tries):
        m = rng.randint(*m_range)
        pts = [(rng.uniform(lo, hi), rng.uniform(lo, hi)) for _ in range(m)]
        hull = MultiPoint(pts).convex_hull
        if hull.geom_type != "Polygon":
            continue
        coords = round_coords(list(hull.exterior.coords)[:-1], coord_type)
        poly = Polygon(coords)
        ok, _ = check_validity(poly, vmin, vmax, lo, hi)
        if ok and is_convex(poly):
            return poly
    return None


# Concave generator: radial / star method.
def gen_concave(rng, vmin, vmax, lo, hi, coord_type, max_tries=3000):
    span = hi - lo
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        cx = rng.uniform(lo + 0.25 * span, hi - 0.25 * span)
        cy = rng.uniform(lo + 0.25 * span, hi - 0.25 * span)
        r_max = rng.uniform(0.10 * span, 0.22 * span)
        ratio = rng.uniform(0.4, 0.8)          # mild concavity band
        r_min = ratio * r_max
        angles = sorted(rng.uniform(0, 2 * math.pi) for _ in range(n))
        coords = []
        for theta in angles:
            r = rng.uniform(r_min, r_max)
            coords.append((cx + r * math.cos(theta), cy + r * math.sin(theta)))
        coords = round_coords(coords, coord_type)
        poly = Polygon(coords)
        ok, _ = check_validity(poly, vmin, vmax, lo, hi)
        if ok and not is_convex(poly):         # must be genuinely concave
            return poly
    return None


# Irregular generator: radial base + stretch + rotate + place.
def gen_irregular(rng, vmin, vmax, lo, hi, coord_type, max_tries=3000):
    span = hi - lo
    for _ in range(max_tries):
        base = gen_concave(rng, vmin, vmax, lo, hi, "float_2dp")
        if base is None:
            continue
        factor = rng.uniform(2.0, 5.0)
        if rng.random() < 0.5:
            s = affinity.scale(base, xfact=factor, yfact=1.0, origin="centroid")
        else:
            s = affinity.scale(base, xfact=1.0, yfact=factor, origin="centroid")
        minx, miny, maxx, maxy = s.bounds
        longest = max(maxx - minx, maxy - miny)
        if longest > 0.9 * span:
            k = (0.9 * span) / longest
            s = affinity.scale(s, xfact=k, yfact=k, origin="centroid")
        s = affinity.rotate(s, rng.uniform(0, 360), origin="centroid")
        minx, miny, maxx, maxy = s.bounds
        w, h = maxx - minx, maxy - miny
        if w > span or h > span:
            continue
        new_minx = rng.uniform(lo, hi - w)
        new_miny = rng.uniform(lo, hi - h)
        s = affinity.translate(s, xoff=new_minx - minx, yoff=new_miny - miny)
        coords = round_coords(list(s.exterior.coords)[:-1], coord_type)
        poly = Polygon(coords)
        ok, _ = check_validity(poly, vmin, vmax, lo, hi)
        if ok:
            return poly
    return None


print("Generators defined.")

In [ ]:
# Valtr algorithm: build a convex polygon with EXACTLY n vertices.
# Used for the hard tier, where random hulls can't reliably reach 20-40 verts.
def _valtr_unit(rng, n):
    """Return n edge vectors (dx, dy) that sum to zero, sorted by angle."""
    def random_steps(k):
        xs = sorted(rng.random() for _ in range(k))
        lo, hi, last_lo, last_hi = [], [], xs[0], xs[0]
        for x in xs[1:-1]:
            if rng.random() < 0.5:
                lo.append(x - last_lo); last_lo = x
            else:
                hi.append(last_hi - x); last_hi = x
        lo.append(xs[-1] - last_lo)
        hi.append(last_hi - xs[-1])
        return lo + hi

    dxs = random_steps(n)
    dys = random_steps(n)
    rng.shuffle(dys)
    vectors = list(zip(dxs, dys))
    vectors.sort(key=lambda v: math.atan2(v[1], v[0]))
    return vectors


def gen_convex_valtr(rng, vmin, vmax, lo, hi, coord_type, max_tries=3000):
    span = hi - lo
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        vectors = _valtr_unit(rng, n)
        x = y = 0.0
        pts = []
        for dx, dy in vectors:
            pts.append((x, y))
            x += dx; y += dy
        minx = min(p[0] for p in pts); miny = min(p[1] for p in pts)
        maxx = max(p[0] for p in pts); maxy = max(p[1] for p in pts)
        w = maxx - minx or 1.0
        h = maxy - miny or 1.0
        target = rng.uniform(0.5 * span, 0.85 * span)
        scale = target / max(w, h)
        pts = [((px - minx) * scale, (py - miny) * scale) for px, py in pts]
        pw = (maxx - minx) * scale
        ph = (maxy - miny) * scale
        ox = rng.uniform(lo, hi - pw) if hi - pw > lo else lo
        oy = rng.uniform(lo, hi - ph) if hi - ph > lo else lo
        pts = [(px + ox, py + oy) for px, py in pts]
        coords = round_coords(pts, coord_type)
        poly = Polygon(coords)
        ok, _ = check_validity(poly, vmin, vmax, lo, hi)
        if ok and is_convex(poly):
            return poly
    return None


# Dispatcher: pick the right generator for a (shape_type, tier) combination.
def generate_one(rng, shape_type, tier):
    vmin, vmax, lo, hi, ct, mr = TIERS[tier]
    if shape_type == "convex":
        if tier == "hard":
            return gen_convex_valtr(rng, vmin, vmax, lo, hi, ct)
        return gen_convex(rng, vmin, vmax, lo, hi, ct, mr)
    elif shape_type == "concave":
        return gen_concave(rng, vmin, vmax, lo, hi, ct)
    else:  # irregular
        return gen_irregular(rng, vmin, vmax, lo, hi, ct)


# Helper: simple-tier convex polygon with an EXACT vertex count,
# used to guarantee a few triangles and quads for shape coverage.
def gen_convex_exact(rng, target_n, lo, hi, coord_type, max_tries=5000):
    for _ in range(max_tries):
        m = rng.randint(target_n, target_n + 15)
        pts = [(rng.uniform(lo, hi), rng.uniform(lo, hi)) for _ in range(m)]
        hull = MultiPoint(pts).convex_hull
        if hull.geom_type != "Polygon":
            continue
        coords = round_coords(list(hull.exterior.coords)[:-1], coord_type)
        if len(coords) != target_n:
            continue
        poly = Polygon(coords)
        ok, _ = check_validity(poly, target_n, target_n, lo, hi)
        if ok and is_convex(poly):
            return poly
    return None


print("Valtr, dispatcher, and exact-vertex helper defined.")

## 3. Ground truth (9 properties) + record builder

The 7 pilot properties plus the 2 new Experiment-1 properties:

- **aspect_ratio** = bbox width / bbox height (global, rel-error eval)
- **edge_length_variance** = variance of all edge lengths (global, rel-error eval)

Both are orientation-independent, so they are computed in **step 1** alongside
area / perimeter / convex (before the random orientation reversal).

Order matters (Section 3.2): compute all non-orientation properties → reverse
the coordinate order with probability 0.5 → serialize to WKT → read orientation
ground truth from the (possibly reversed) ring.


In [ ]:
# Step 1: properties that do NOT depend on winding direction (now 8 of them).
# Also returns the bbox diagonal, used later to normalize centroid/bbox errors.
def compute_orientation_independent(poly):
    minx, miny, maxx, maxy = poly.bounds
    cx, cy = poly.centroid.coords[0]

    # --- 2 new Experiment-1 properties (both orientation-independent) ---
    ring = list(poly.exterior.coords)            # closed ring (last == first)
    edge_lengths = [math.dist(ring[i], ring[i + 1]) for i in range(len(ring) - 1)]
    n_e = len(edge_lengths)
    mean_e = sum(edge_lengths) / n_e
    edge_var = sum((L - mean_e) ** 2 for L in edge_lengths) / n_e
    w, h = maxx - minx, maxy - miny
    aspect = w / h if h else float("inf")

    props = {
        "vertex_count": len(poly.exterior.coords) - 1,
        "bbox": [round(minx, 4), round(miny, 4),
                 round(maxx, 4), round(maxy, 4)],
        "centroid": [round(cx, 4), round(cy, 4)],
        "area": round(poly.area, 4),
        "perimeter": round(poly.length, 4),
        "convex": bool(is_convex(poly)),
        "aspect_ratio": round(aspect, 4),
        "edge_length_variance": round(edge_var, 4),
    }
    bbox_diag = math.dist((minx, miny), (maxx, maxy))
    return props, bbox_diag


# Step 2: reverse the coordinate order with probability 0.5.
def maybe_reverse(poly, rng):
    if rng.random() < 0.5:
        coords = list(poly.exterior.coords)[::-1]
        return Polygon(coords), True
    return poly, False


# Step 3: serialize to WKT, formatting coords to match the tier.
def to_wkt(poly, coord_type):
    coords = list(poly.exterior.coords)        # includes closing coord
    parts = []
    for x, y in coords:
        if coord_type == "integer":
            parts.append(f"{int(round(x))} {int(round(y))}")
        else:
            parts.append(f"{x:.2f} {y:.2f}")
    return "POLYGON((" + ", ".join(parts) + "))"


# Step 4: read orientation from the (possibly reversed) ring.
def get_orientation(poly):
    return "ccw" if shapely.is_ccw(poly.exterior) else "cw"


# Build one full dataset record, enforcing the correct step order.
def build_record(poly, tier, shape_type, index, coord_type, rng, seed):
    props, bbox_diag = compute_orientation_independent(poly)   # step 1
    poly2, was_reversed = maybe_reverse(poly, rng)             # step 2
    wkt = to_wkt(poly2, coord_type)                            # step 3
    props["orientation"] = get_orientation(poly2)              # step 4

    return {
        "object_id": f"poly_{tier}_{shape_type}_{index:03d}",
        "tier": tier,
        "shape_type": shape_type,
        "num_vertices": props["vertex_count"],
        "wkt": wkt,
        "properties": props,
        "metadata": {
            "coordinate_type": coord_type,
            "is_valid": bool(poly2.is_valid),
            "is_simple": bool(poly2.is_simple),
            "orientation_was_reversed": was_reversed,
            "random_seed": seed,
            "bbox_diagonal": round(bbox_diag, 4),
        },
    }


print("Ground-truth and record builder defined (9 properties).")

## 4. Build the 300-polygon dataset + summary

300 polygons = 3 tiers × 100. Within each tier the 3 shape categories are split
**34 / 33 / 33** (convex / concave / irregular) — approximately balanced, as the
PDF allows. For the simple-convex group a few triangles and quads are forced for
low-vertex coverage.


In [ ]:
# Per-tier shape split: 34 + 33 + 33 = 100 polygons per tier.
PER_SHAPE = {"convex": 34, "concave": 33, "irregular": 33}


def build_dataset(seed=42):
    rng = random.Random(seed)
    records = []
    for tier in ["simple", "medium", "hard"]:
        vmin, vmax, lo, hi, ct, mr = TIERS[tier]
        for shape_type in ["convex", "concave", "irregular"]:
            for i in range(1, PER_SHAPE[shape_type] + 1):     # 001..0NN
                poly = None
                # simple-tier convex: force a couple of triangles + quads
                if tier == "simple" and shape_type == "convex":
                    if i in (1, 2):
                        poly = gen_convex_exact(rng, 3, lo, hi, ct)
                    elif i in (3, 4):
                        poly = gen_convex_exact(rng, 4, lo, hi, ct)
                if poly is None:
                    poly = generate_one(rng, shape_type, tier)
                if poly is None:
                    raise RuntimeError(f"Failed to generate {tier}/{shape_type} #{i}")
                records.append(build_record(poly, tier, shape_type, i, ct, rng, seed))
    return records


# Min/max/mean/median/std for a list of numbers.
def stats_for(values):
    n = len(values)
    mean = sum(values) / n
    sv = sorted(values)
    median = sv[n // 2] if n % 2 else (sv[n // 2 - 1] + sv[n // 2]) / 2
    var = sum((v - mean) ** 2 for v in values) / n
    return {"min": round(min(values), 2), "max": round(max(values), 2),
            "mean": round(mean, 2), "median": round(median, 2),
            "std": round(var ** 0.5, 2)}


# Section 7 summary statistics (extended with the 2 new properties).
def summarize(records):
    summary = {"total": len(records)}
    by_tier = {}
    for r in records:
        by_tier.setdefault(r["tier"], {}).setdefault(r["shape_type"], 0)
        by_tier[r["tier"]][r["shape_type"]] += 1
    summary["counts_by_tier_shape"] = by_tier
    summary["convex_overall"] = sum(1 for r in records if r["properties"]["convex"])

    dist = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        dist[tier] = {
            "vertex_count": stats_for([r["num_vertices"] for r in rs]),
            "area": stats_for([r["properties"]["area"] for r in rs]),
            "perimeter": stats_for([r["properties"]["perimeter"] for r in rs]),
            "bbox_diagonal": stats_for([r["metadata"]["bbox_diagonal"] for r in rs]),
            "aspect_ratio": stats_for([r["properties"]["aspect_ratio"] for r in rs]),
            "edge_length_variance": stats_for([r["properties"]["edge_length_variance"] for r in rs]),
            "wkt_length": stats_for([len(r["wkt"]) for r in rs]),
        }
    summary["distribution_by_tier"] = dist

    orient = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        cw = sum(1 for r in rs if r["properties"]["orientation"] == "cw")
        orient[tier] = {"cw": cw, "ccw": len(rs) - cw}
    summary["orientation_balance"] = orient
    return summary


# Run it all.
SEED = 42
print("Generating 300-polygon dataset (seed =", SEED, ")...")
records = build_dataset(SEED)

with open("geometry_exp1_dataset.json", "w") as f:
    json.dump(records, f, indent=2)
print("Saved geometry_exp1_dataset.json with", len(records), "polygons.")

summary = summarize(records)
with open("geometry_exp1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved geometry_exp1_summary.json.\n")

print("Total polygons:", summary["total"])
print("Convex overall:", summary["convex_overall"],
      f"({100*summary['convex_overall']/summary['total']:.0f}%)")
print("\nCounts by tier/shape:")
for tier, shapes in summary["counts_by_tier_shape"].items():
    print(f"  {tier}: {shapes}")
print("\nOrientation balance:")
for tier, ob in summary["orientation_balance"].items():
    print(f"  {tier}: {ob}")
print("\nVertex-count range per tier:")
for tier, d in summary["distribution_by_tier"].items():
    vc = d["vertex_count"]
    print(f"  {tier}: min {vc['min']}, max {vc['max']}, mean {vc['mean']}")

## 5. Independent verification

Re-derive every property from the raw WKT coordinates with an **independent**
implementation (not Shapely), and confirm it matches the stored ground truth.
This catches any bug in the generator or the ground-truth step.


In [ ]:
import re

def parse_wkt(wkt):
    nums = re.findall(r"-?\d+(?:\.\d+)?", wkt)
    vals = list(map(float, nums))
    pts = [(vals[i], vals[i + 1]) for i in range(0, len(vals), 2)]
    return pts  # closed ring (last == first)

def shoelace_area(pts):
    s = 0.0
    for i in range(len(pts) - 1):
        x1, y1 = pts[i]; x2, y2 = pts[i + 1]
        s += x1 * y2 - x2 * y1
    return abs(s) / 2.0

def perim(pts):
    return sum(math.dist(pts[i], pts[i + 1]) for i in range(len(pts) - 1))

def aspect_from(pts):
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    w = max(xs) - min(xs); h = max(ys) - min(ys)
    return w / h if h else float("inf")

def edgevar_from(pts):
    e = [math.dist(pts[i], pts[i + 1]) for i in range(len(pts) - 1)]
    m = sum(e) / len(e)
    return sum((L - m) ** 2 for L in e) / len(e)

mism = 0
for r in records:
    pts = parse_wkt(r["wkt"])
    gt = r["properties"]
    checks = {
        "vertex_count": (len(pts) - 1, gt["vertex_count"], 0),
        "area": (shoelace_area(pts), gt["area"], 1.0),
        "perimeter": (perim(pts), gt["perimeter"], 0.5),
        "aspect_ratio": (aspect_from(pts), gt["aspect_ratio"], 0.02),
        "edge_length_variance": (edgevar_from(pts), gt["edge_length_variance"], max(1.0, 0.02*gt["edge_length_variance"])),
    }
    for name, (got, exp, tol) in checks.items():
        if abs(got - exp) > tol:
            mism += 1
            if mism <= 10:
                print(f"MISMATCH {r['object_id']} {name}: indep={got:.3f} stored={exp:.3f}")

print(f"\nIndependent verification: {mism} mismatches across {len(records)} polygons.")
print("(small area/perimeter gaps come from coordinate rounding -- tolerances applied)")

# validity sanity
bad = [r["object_id"] for r in records
       if not r["metadata"]["is_valid"] or not r["metadata"]["is_simple"]]
print("Invalid / non-simple polygons:", len(bad))
print("Properties present:", list(records[0]["properties"].keys()))

## 6. Visual spot-check

Plot one polygon per (tier × shape) — 9 polygons — for an eyeball check that
convex shapes look convex, concave shapes have dents, and irregular shapes are
stretched.


In [ ]:
def find(tier, shape):
    for r in records:
        if r["tier"] == tier and r["shape_type"] == shape:
            return r
    return None

fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for row, tier in enumerate(["simple", "medium", "hard"]):
    for col, shape in enumerate(["convex", "concave", "irregular"]):
        ax = axes[row][col]
        r = find(tier, shape)
        pts = parse_wkt(r["wkt"])
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        ax.plot(xs, ys, "-o", ms=2)
        ax.fill(xs, ys, alpha=0.25)
        ax.set_title(f"{tier}/{shape}\nv={r['num_vertices']} conv={r['properties']['convex']}",
                     fontsize=9)
        ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("spotcheck_exp1.png", dpi=110)
plt.show()
print("Saved spotcheck_exp1.png")

## 7. Download (Colab)

Download the dataset, summary, and spot-check figure. (Skip if running locally —
the files are already saved in the working directory.)


In [ ]:
try:
    from google.colab import files
    files.download("geometry_exp1_dataset.json")
    files.download("geometry_exp1_summary.json")
    files.download("spotcheck_exp1.png")
except Exception as e:
    print("Not on Colab (files saved locally):", e)